In [53]:
source('/home/jupyter/workspaces/temporaldigitalhealthphenotyping/across_cohort_descriptive_func.R')
source('/home/jupyter/workspaces/temporaldigitalhealthphenotyping/ind_descriptive_func.R')
source('/home/jupyter/workspaces/temporaldigitalhealthphenotyping/cci_function.R')


path.in <- '/home/jupyter/workspaces/Data/'

In [54]:
diabetes.activity.pid <- read.delim(
    paste0(path.in, 'Diabetes_activity_pid_filtered.txt'),
    as.is = T,
    stringsAsFactors = F
    );
diabetes.sleep.pid <- read.delim(
    paste0(path.in, 'Diabetes_sleep_pid_filtered.txt'),
    as.is = T,
    stringsAsFactors = F
    );
diabetes.hr.pid <- read.delim(
    paste0(path.in, 'Diabetes_hr_pid_filtered.txt'),
    as.is = T,
    stringsAsFactors = F
    );

insomnia.activity.pid <- read.delim(
    paste0(path.in, 'Insomnia_activity_pid_filtered.txt'),
    as.is = T,
    stringsAsFactors = F
    );
insomnia.sleep.pid <- read.delim(
    paste0(path.in, 'Insomnia_sleep_pid_filtered.txt'),
    as.is = T,
    stringsAsFactors = F
    );
insomnia.hr.pid <- read.delim(
    paste0(path.in, 'Insomnia_hr_pid_filtered.txt'),
    as.is = T,
    stringsAsFactors = F
    );

prostate.activity.pid <- read.delim(
    paste0(path.in, 'Prostate_activity_pid_filtered.txt'),
    as.is = T,
    stringsAsFactors = F
    );
prostate.sleep.pid <- read.delim(
    paste0(path.in, 'Prostate_sleep_pid_filtered.txt'),
    as.is = T,
    stringsAsFactors = F
    );
prostate.hr.pid <- read.delim(
    paste0(path.in, 'Prostate_hr_pid_filtered.txt'),
    as.is = T,
    stringsAsFactors = F
    );

healthy.activity.pid <- read.delim(
    paste0(path.in, 'Healthy_activity_pid_filtered.txt'),
    as.is = T,
    stringsAsFactors = F
    );
healthy.sleep.pid <- read.delim(
    paste0(path.in, 'Healthy_sleep_pid_filtered.txt'),
    as.is = T,
    stringsAsFactors = F
    );
healthy.hr.pid <- read.delim(
    paste0(path.in, 'Healthy_hr_pid_filtered.txt'),
    as.is = T,
    stringsAsFactors = F
    );

In [55]:
diabetes.pid <- Reduce(intersect, list(diabetes.activity.pid$x, diabetes.sleep.pid$x, diabetes.hr.pid$x))
length(diabetes.pid)
write.table(diabetes.pid, paste0(path.in, 'diabetes_final_pid.txt'), sep = '\t', quote = F, row.names = F)

insomnia.pid <- Reduce(intersect, list(insomnia.activity.pid$x, insomnia.sleep.pid$x, insomnia.hr.pid$x))
length(insomnia.pid)
write.table(insomnia.pid, paste0(path.in, 'insomnia_final_pid.txt'), sep = '\t', quote = F, row.names = F)

prostate.pid <- Reduce(intersect, list(prostate.activity.pid$x, prostate.sleep.pid$x, prostate.hr.pid$x))
length(prostate.pid)
write.table(prostate.pid, paste0(path.in, 'prostate_final_pid.txt'), sep = '\t', quote = F, row.names = F)

healthy.pid <- Reduce(intersect, list(healthy.activity.pid$x, healthy.sleep.pid$x, healthy.hr.pid$x))
length(healthy.pid)
write.table(healthy.pid, paste0(path.in, 'healthy_final_pid.txt'), sep = '\t', quote = F, row.names = F)


[1] 1547

[1] 1625

[1] 281

[1] 6960

In [13]:
# read data 
library(tidyverse)
library(bigrquery)

diabetes.pid.sql <- paste(diabetes.pid, collapse = ", ")
diabetes_meta_sql <- paste("
    SELECT
        person.person_id,
        person.birth_datetime as date_of_birth,
        person.race_concept_id,
        p_race_concept.concept_name as race,
        person.ethnicity_concept_id,
        p_ethnicity_concept.concept_name as ethnicity,
        person.sex_at_birth_concept_id,
        p_sex_at_birth_concept.concept_name as sex_at_birth,
        person.self_reported_category_concept_id,
        p_self_reported_category_concept.concept_name as self_reported_category 
    FROM
        `person` person 
    LEFT JOIN
        `concept` p_race_concept 
            ON person.race_concept_id = p_race_concept.concept_id 
    LEFT JOIN
        `concept` p_ethnicity_concept 
            ON person.ethnicity_concept_id = p_ethnicity_concept.concept_id 
    LEFT JOIN
        `concept` p_sex_at_birth_concept 
            ON person.sex_at_birth_concept_id = p_sex_at_birth_concept.concept_id 
    LEFT JOIN
        `concept` p_self_reported_category_concept 
            ON person.self_reported_category_concept_id = p_self_reported_category_concept.concept_id  
    WHERE
        person.person_id IN (", diabetes.pid.sql, ")"
)
diabetes_meta_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "diabetes_meta",
  "diabetes_meta_*.csv")
message(str_glue('The data will be written to {diabetes_meta_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), diabetes_meta_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  diabetes_meta_path,
  destination_format = "CSV")


The data will be written to gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260505/diabetes_meta/diabetes_meta_*.csv. Use this path when reading the data into your notebooks in the future.



In [4]:
library(tidyverse)
library(bigrquery)

read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(race = col_character(), ethnicity = col_character(), sex_at_birth = col_character(), self_reported_category = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
diabetes_meta_path <- 'gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260505/diabetes_meta/diabetes_meta_*.csv'

diabetes_meta <- read_bq_export_from_workspace_bucket(diabetes_meta_path)

dim(diabetes_meta)

head(diabetes_meta, 5) 

Loading gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260505/diabetes_meta/diabetes_meta_000000000000.csv.



[1] 1547   10

person_id,date_of_birth,race_concept_id,race,ethnicity_concept_id,ethnicity,sex_at_birth_concept_id,sex_at_birth,self_reported_category_concept_id,self_reported_category
<dbl>,<chr>,<dbl>,<chr>,<dbl>,<chr>,<dbl>,<chr>,<dbl>,<chr>
1344748,1966-06-15 00:00:00 UTC,2e+09,More than one population,38003563,Hispanic or Latino,45878463,Female,2e+09,More than one population
1617237,1964-06-15 00:00:00 UTC,2e+09,More than one population,38003563,Hispanic or Latino,45878463,Female,2e+09,More than one population
2977576,1968-06-15 00:00:00 UTC,2e+09,More than one population,38003563,Hispanic or Latino,45878463,Female,2e+09,More than one population
3471584,1958-06-15 00:00:00 UTC,2e+09,More than one population,38003563,Hispanic or Latino,45878463,Female,2e+09,More than one population
9004048,1990-06-15 00:00:00 UTC,2e+09,More than one population,38003563,Hispanic or Latino,45878463,Female,2e+09,More than one population


In [15]:
demographic_tbl_func(df = diabetes_meta)

# A tibble: 21 × 2
   Demographic                      Value        
   <chr>                            <chr>        
 1 N                                "1,547"      
 2 Age                              ""           
 3 18-44                            "160 (10%)"  
 4 45-64                            "577 (37%)"  
 5 >65                              "810 (52%)"  
 6 Ethnicity                        ""           
 7 Hispanic or Latino               "135 (9%)"   
 8 Not Hispanic or Latino           "1,371 (89%)"
 9 Others                           "41 (3%)"    
10 Race                             ""           
11 American Indian or Alaska Native "12 (1%)"    
12 Asian                            "45 (3%)"    
13 Black or African American        "158 (10%)"  
14 Middle Eastern or North African  "5 (<1%)"    
15 More than one population         "105 (7%)"   
16 Others                           "123 (8%)"   
17 White                            "1,099 (71%)"
18 Sex at birth                

NULL

In [16]:
table(diabetes_meta$ethnicity_concept_id)


  903079   903096  1586148 38003563 38003564 
       1       21       19      135     1371 

In [17]:
# INSOMNIA read data 
library(tidyverse)
library(bigrquery)

insomnia.pid.sql <- paste(insomnia.pid, collapse = ", ")
insomnia_meta_sql <- paste("
    SELECT
        person.person_id,
        person.birth_datetime as date_of_birth,
        person.race_concept_id,
        p_race_concept.concept_name as race,
        person.ethnicity_concept_id,
        p_ethnicity_concept.concept_name as ethnicity,
        person.sex_at_birth_concept_id,
        p_sex_at_birth_concept.concept_name as sex_at_birth,
        person.self_reported_category_concept_id,
        p_self_reported_category_concept.concept_name as self_reported_category 
    FROM
        `person` person 
    LEFT JOIN
        `concept` p_race_concept 
            ON person.race_concept_id = p_race_concept.concept_id 
    LEFT JOIN
        `concept` p_ethnicity_concept 
            ON person.ethnicity_concept_id = p_ethnicity_concept.concept_id 
    LEFT JOIN
        `concept` p_sex_at_birth_concept 
            ON person.sex_at_birth_concept_id = p_sex_at_birth_concept.concept_id 
    LEFT JOIN
        `concept` p_self_reported_category_concept 
            ON person.self_reported_category_concept_id = p_self_reported_category_concept.concept_id  
    WHERE
        person.person_id IN (", insomnia.pid.sql, ")"
)
insomnia_meta_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "insomnia_meta",
  "insomnia_meta_*.csv")
message(str_glue('The data will be written to {insomnia_meta_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), insomnia_meta_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  insomnia_meta_path,
  destination_format = "CSV")


The data will be written to gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260505/insomnia_meta/insomnia_meta_*.csv. Use this path when reading the data into your notebooks in the future.



In [5]:
library(tidyverse)
library(bigrquery)

read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(race = col_character(), ethnicity = col_character(), sex_at_birth = col_character(), self_reported_category = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
insomnia_meta_path <- 'gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260505/insomnia_meta/insomnia_meta_*.csv'
insomnia_meta <- read_bq_export_from_workspace_bucket(insomnia_meta_path)

dim(insomnia_meta)

head(insomnia_meta, 5) 

Loading gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260505/insomnia_meta/insomnia_meta_000000000000.csv.



[1] 1625   10

person_id,date_of_birth,race_concept_id,race,ethnicity_concept_id,ethnicity,sex_at_birth_concept_id,sex_at_birth,self_reported_category_concept_id,self_reported_category
<dbl>,<chr>,<dbl>,<chr>,<dbl>,<chr>,<dbl>,<chr>,<dbl>,<chr>
1107258,1942-06-15 00:00:00 UTC,2.1e+09,None Indicated,38003563,Hispanic or Latino,45880669,Male,1586147,What Race Ethnicity: Hispanic
1598808,1959-06-15 00:00:00 UTC,2.1e+09,None Indicated,38003563,Hispanic or Latino,45880669,Male,1586147,What Race Ethnicity: Hispanic
4534876,1995-06-15 00:00:00 UTC,2.1e+09,None Indicated,38003563,Hispanic or Latino,45880669,Male,1586147,What Race Ethnicity: Hispanic
2812244,1976-06-15 00:00:00 UTC,2.1e+09,None Indicated,38003563,Hispanic or Latino,45880669,Male,1586147,What Race Ethnicity: Hispanic
2618163,1981-06-15 00:00:00 UTC,2.1e+09,None Indicated,38003563,Hispanic or Latino,45880669,Male,1586147,What Race Ethnicity: Hispanic


In [26]:
demographic_tbl_func(df = insomnia_meta)

# A tibble: 21 × 2
   Demographic                      Value        
   <chr>                            <chr>        
 1 N                                "1,625"      
 2 Age                              ""           
 3 18-44                            "278 (17%)"  
 4 45-64                            "539 (33%)"  
 5 >65                              "808 (50%)"  
 6 Ethnicity                        ""           
 7 Hispanic or Latino               "119 (7%)"   
 8 Not Hispanic or Latino           "1,478 (91%)"
 9 Others                           "28 (2%)"    
10 Race                             ""           
11 American Indian or Alaska Native "11 (1%)"    
12 Asian                            "27 (2%)"    
13 Black or African American        "97 (6%)"    
14 Middle Eastern or North African  "8 (<1%)"    
15 More than one population         "88 (5%)"    
16 Others                           "104 (6%)"   
17 White                            "1,290 (79%)"
18 Sex at birth                

NULL

In [27]:
# INSOMNIA read data 
library(tidyverse)
library(bigrquery)

pca.pid.sql <- paste(prostate.pid, collapse = ", ")
pca_meta_sql <- paste("
    SELECT
        person.person_id,
        person.birth_datetime as date_of_birth,
        person.race_concept_id,
        p_race_concept.concept_name as race,
        person.ethnicity_concept_id,
        p_ethnicity_concept.concept_name as ethnicity,
        person.sex_at_birth_concept_id,
        p_sex_at_birth_concept.concept_name as sex_at_birth,
        person.self_reported_category_concept_id,
        p_self_reported_category_concept.concept_name as self_reported_category 
    FROM
        `person` person 
    LEFT JOIN
        `concept` p_race_concept 
            ON person.race_concept_id = p_race_concept.concept_id 
    LEFT JOIN
        `concept` p_ethnicity_concept 
            ON person.ethnicity_concept_id = p_ethnicity_concept.concept_id 
    LEFT JOIN
        `concept` p_sex_at_birth_concept 
            ON person.sex_at_birth_concept_id = p_sex_at_birth_concept.concept_id 
    LEFT JOIN
        `concept` p_self_reported_category_concept 
            ON person.self_reported_category_concept_id = p_self_reported_category_concept.concept_id  
    WHERE
        person.person_id IN (", pca.pid.sql, ")"
)
pca_meta_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "pca_meta",
  "pca_meta_*.csv")
message(str_glue('The data will be written to {pca_meta_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), pca_meta_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  pca_meta_path,
  destination_format = "CSV")


The data will be written to gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260505/pca_meta/pca_meta_*.csv. Use this path when reading the data into your notebooks in the future.



In [6]:
library(tidyverse)
library(bigrquery)

read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(race = col_character(), ethnicity = col_character(), sex_at_birth = col_character(), self_reported_category = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
pca_meta_path <- 'gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260505/pca_meta/pca_meta_*.csv'
pca_meta <- read_bq_export_from_workspace_bucket(pca_meta_path)

dim(pca_meta)

head(pca_meta, 5)

Loading gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260505/pca_meta/pca_meta_000000000000.csv.



[1] 281  10

person_id,date_of_birth,race_concept_id,race,ethnicity_concept_id,ethnicity,sex_at_birth_concept_id,sex_at_birth,self_reported_category_concept_id,self_reported_category
<dbl>,<chr>,<dbl>,<chr>,<dbl>,<chr>,<dbl>,<chr>,<dbl>,<chr>
2967502,1953-06-15 00:00:00 UTC,45882607,None of these,1586148,What Race Ethnicity: Race Ethnicity None Of These,45880669,Male,45882607,None of these
2462406,1948-06-15 00:00:00 UTC,8527,White,38003563,Hispanic or Latino,45880669,Male,2000000008,More than one population
7845067,1961-06-15 00:00:00 UTC,8527,White,38003563,Hispanic or Latino,45880669,Male,2000000008,More than one population
7047614,1949-06-15 00:00:00 UTC,8657,American Indian or Alaska Native,38003563,Hispanic or Latino,45880669,Male,2000000008,More than one population
3849859,1963-06-15 00:00:00 UTC,2000000008,More than one population,38003564,Not Hispanic or Latino,45880669,Male,2000000008,More than one population


In [29]:
demographic_tbl_func(df = pca_meta)

# A tibble: 18 × 2
   Demographic                      Value       
   <chr>                            <chr>       
 1 N                                "281"       
 2 Age                              ""          
 3 45-64                            "11 (4%)"   
 4 >65                              "270 (96%)" 
 5 Ethnicity                        ""          
 6 Hispanic or Latino               "10 (4%)"   
 7 Not Hispanic or Latino           "267 (95%)" 
 8 Others                           "4 (1%)"    
 9 Race                             ""          
10 American Indian or Alaska Native "2 (1%)"    
11 Asian                            "3 (1%)"    
12 Black or African American        "12 (4%)"   
13 Middle Eastern or North African  "1 (<1%)"   
14 More than one population         "5 (2%)"    
15 Others                           "11 (4%)"   
16 White                            "247 (88%)" 
17 Sex at birth                     ""          
18 Male                             "281 (100%)"


NULL

In [30]:
# INSOMNIA read data 
library(tidyverse)
library(bigrquery)

healthy.pid.sql <- paste(healthy.pid, collapse = ", ")
healthy_meta_sql <- paste("
    SELECT
        person.person_id,
        person.birth_datetime as date_of_birth,
        person.race_concept_id,
        p_race_concept.concept_name as race,
        person.ethnicity_concept_id,
        p_ethnicity_concept.concept_name as ethnicity,
        person.sex_at_birth_concept_id,
        p_sex_at_birth_concept.concept_name as sex_at_birth,
        person.self_reported_category_concept_id,
        p_self_reported_category_concept.concept_name as self_reported_category 
    FROM
        `person` person 
    LEFT JOIN
        `concept` p_race_concept 
            ON person.race_concept_id = p_race_concept.concept_id 
    LEFT JOIN
        `concept` p_ethnicity_concept 
            ON person.ethnicity_concept_id = p_ethnicity_concept.concept_id 
    LEFT JOIN
        `concept` p_sex_at_birth_concept 
            ON person.sex_at_birth_concept_id = p_sex_at_birth_concept.concept_id 
    LEFT JOIN
        `concept` p_self_reported_category_concept 
            ON person.self_reported_category_concept_id = p_self_reported_category_concept.concept_id  
    WHERE
        person.person_id IN (", healthy.pid.sql, ")"
)
healthy_meta_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "healthy_meta",
  "healthy_meta_*.csv")
message(str_glue('The data will be written to {healthy_meta_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), healthy_meta_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  healthy_meta_path,
  destination_format = "CSV")


The data will be written to gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260505/healthy_meta/healthy_meta_*.csv. Use this path when reading the data into your notebooks in the future.



In [56]:
library(tidyverse)
library(bigrquery)

read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(race = col_character(), ethnicity = col_character(), sex_at_birth = col_character(), self_reported_category = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
healthy_meta_path <- 'gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260505/healthy_meta/healthy_meta_*.csv'

healthy_meta <- read_bq_export_from_workspace_bucket(healthy_meta_path)

dim(healthy_meta)

head(healthy_meta, 5)

Loading gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260505/healthy_meta/healthy_meta_000000000000.csv.



[1] 6960   10

person_id,date_of_birth,race_concept_id,race,ethnicity_concept_id,ethnicity,sex_at_birth_concept_id,sex_at_birth,self_reported_category_concept_id,self_reported_category
<dbl>,<chr>,<dbl>,<chr>,<dbl>,<chr>,<dbl>,<chr>,<dbl>,<chr>
4337953,1965-06-15 00:00:00 UTC,8557,Native Hawaiian or Other Pacific Islander,38003564,Not Hispanic or Latino,45878463,Female,8557,Native Hawaiian or Other Pacific Islander
1188079,1981-06-15 00:00:00 UTC,45882607,None of these,1586148,What Race Ethnicity: Race Ethnicity None Of These,45878463,Female,45882607,None of these
1346705,1998-06-15 00:00:00 UTC,45882607,None of these,1586148,What Race Ethnicity: Race Ethnicity None Of These,45878463,Female,45882607,None of these
3043509,1997-06-15 00:00:00 UTC,45882607,None of these,1586148,What Race Ethnicity: Race Ethnicity None Of These,45878463,Female,45882607,None of these
2523178,1992-06-15 00:00:00 UTC,45882607,None of these,1586148,What Race Ethnicity: Race Ethnicity None Of These,45878463,Female,45882607,None of these


In [32]:
demographic_tbl_func(df = healthy_meta)

# A tibble: 21 × 2
   Demographic                      Value        
   <chr>                            <chr>        
 1 N                                "6,960"      
 2 Age                              ""           
 3 18-44                            "1,928 (28%)"
 4 45-64                            "2,294 (33%)"
 5 >65                              "2,738 (39%)"
 6 Ethnicity                        ""           
 7 Hispanic or Latino               "840 (12%)"  
 8 Not Hispanic or Latino           "5,959 (86%)"
 9 Others                           "161 (2%)"   
10 Race                             ""           
11 American Indian or Alaska Native "49 (1%)"    
12 Asian                            "417 (6%)"   
13 Black or African American        "366 (5%)"   
14 Middle Eastern or North African  "34 (<1%)"   
15 More than one population         "513 (7%)"   
16 Others                           "680 (10%)"  
17 White                            "4,901 (70%)"
18 Sex at birth                

NULL

In [33]:
# INSOMNIA read data 
library(tidyverse)
library(bigrquery)

diabetes.pid.sql <- paste(diabetes.pid, collapse = ", ")

diabetes_survey_sql <- paste("
    SELECT
        c_occurrence.person_id,
        c_occurrence.condition_concept_id,
        c_standard_concept.concept_name as standard_concept_name,
        c_standard_concept.concept_code as standard_concept_code,
        c_standard_concept.vocabulary_id as standard_vocabulary,
        c_occurrence.condition_start_datetime,
        c_occurrence.condition_end_datetime,
        c_occurrence.condition_type_concept_id,
        c_type.concept_name as condition_type_concept_name,
        c_occurrence.stop_reason,
        c_occurrence.visit_occurrence_id,
        visit.concept_name as visit_occurrence_concept_name,
        c_occurrence.condition_source_value,
        c_occurrence.condition_source_concept_id,
        c_source_concept.concept_name as source_concept_name,
        c_source_concept.concept_code as source_concept_code,
        c_source_concept.vocabulary_id as source_vocabulary,
        c_occurrence.condition_status_source_value,
        c_occurrence.condition_status_concept_id,
        c_status.concept_name as condition_status_concept_name 
    FROM
        ( SELECT
            * 
        FROM
            `condition_occurrence` c_occurrence 
        WHERE
            (
                condition_concept_id IN (SELECT
                    DISTINCT c.concept_id 
                FROM
                    `cb_criteria` c 
                JOIN
                    (SELECT
                        CAST(cr.id as string) AS id       
                    FROM
                        `cb_criteria` cr       
                    WHERE
                        concept_id IN (195771, 201254, 201820, 201826, 4008576, 4024659, 40482801, 40482883, 40484648, 40484649, 40485020, 4058243, 4063042, 4063043, 4099214, 4099334, 4099651, 4129519, 4130162, 4152858, 4193704, 4200875, 4202383, 4230254, 42537681, 4263902, 4274025, 4304377, 4311629, 43531007, 43531008, 43531009, 43531010, 436670, 443238, 443412, 44808385, 45757474, 45757789)       
                        AND full_text LIKE '%_rank1]%'      ) a 
                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                        OR c.path LIKE CONCAT('%.', a.id) 
                        OR c.path LIKE CONCAT(a.id, '.%') 
                        OR c.path = a.id) 
                WHERE
                    is_standard = 1 
                    AND is_selectable = 1)
                )
            AND 
                c_occurrence.person_id IN (", diabetes.pid.sql, ")
            ) c_occurrence 
        LEFT JOIN
            `concept` c_standard_concept 
                ON c_occurrence.condition_concept_id = c_standard_concept.concept_id 
        LEFT JOIN
            `concept` c_type 
                ON c_occurrence.condition_type_concept_id = c_type.concept_id 
        LEFT JOIN
            `visit_occurrence` v 
                ON c_occurrence.visit_occurrence_id = v.visit_occurrence_id 
        LEFT JOIN
            `concept` visit 
                ON v.visit_concept_id = visit.concept_id 
        LEFT JOIN
            `concept` c_source_concept 
                ON c_occurrence.condition_source_concept_id = c_source_concept.concept_id 
        LEFT JOIN
            `concept` c_status 
                ON c_occurrence.condition_status_concept_id = c_status.concept_id", sep="")


# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
diabetes_survey_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "diabetes_survey",
  "diabetes_survey_*.csv")
message(str_glue('The data will be written to {diabetes_survey_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), diabetes_survey_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  diabetes_survey_path,
  destination_format = "CSV")


The data will be written to gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260505/diabetes_survey/diabetes_survey_*.csv. Use this path when reading the data into your notebooks in the future.



In [8]:
library(tidyverse)
library(bigrquery)

read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(standard_concept_name = col_character(), standard_concept_code = col_character(), standard_vocabulary = col_character(), condition_type_concept_name = col_character(), stop_reason = col_character(), visit_occurrence_concept_name = col_character(), condition_source_value = col_character(), source_concept_name = col_character(), source_concept_code = col_character(), source_vocabulary = col_character(), condition_status_source_value = col_character(), condition_status_concept_name = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
diabetes_survey_path <- 'gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260505/diabetes_survey/diabetes_survey_*.csv'

diabetes_survey <- read_bq_export_from_workspace_bucket(diabetes_survey_path)

dim(diabetes_survey)

head(diabetes_survey, 5)

Loading gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260505/diabetes_survey/diabetes_survey_000000000000.csv.



[1] 866650     20

person_id,condition_concept_id,standard_concept_name,standard_concept_code,standard_vocabulary,condition_start_datetime,condition_end_datetime,condition_type_concept_id,condition_type_concept_name,stop_reason,visit_occurrence_id,visit_occurrence_concept_name,condition_source_value,condition_source_concept_id,source_concept_name,source_concept_code,source_vocabulary,condition_status_source_value,condition_status_concept_id,condition_status_concept_name
<dbl>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<chr>,<chr>,<dbl>,<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>,<dbl>,<chr>
9706700,4174262,Polyneuropathy,42345000,SNOMED,2019-05-06 16:29:47 UTC,2019-05-06 16:29:47 UTC,38000280,Observation recorded from EHR,NA,5.4e+16,Outpatient Visit,G62.9,35207458,"Polyneuropathy, unspecified",G62.9,ICD10CM,NA,NA,NA
1053250,437246,Vitamin B deficiency,47903000,SNOMED,2018-06-20 14:56:00 UTC,NA,5086,Condition tested for by diagnostic procedure,NA,7.0e+15,Outpatient Visit,266.2,44829891,Other B-complex deficiencies,266.2,ICD9CM,NA,NA,NA
2100232,257628,Systemic lupus erythematosus,55464009,SNOMED,2020-06-22 05:30:00 UTC,NA,5086,Condition tested for by diagnostic procedure,NA,7.0e+15,Outpatient Visit,M32.9,35208827,"Systemic lupus erythematosus, unspecified",M32.9,ICD10CM,NA,NA,NA
1958922,320128,Essential hypertension,59621000,SNOMED,2022-06-24 10:12:00 UTC,NA,5086,Condition tested for by diagnostic procedure,NA,7.0e+15,Outpatient Visit,401.9,44821949,Unspecified essential hypertension,401.9,ICD9CM,NA,NA,NA
1053250,320744,Complete atrioventricular block,27885002,SNOMED,2021-03-03 14:16:00 UTC,NA,5086,Condition tested for by diagnostic procedure,NA,7.0e+15,Outpatient Visit,I44.2,35207768,"Atrioventricular block, complete",I44.2,ICD10CM,NA,NA,NA


In [35]:
condition_func(df = diabetes_survey)

[1] 1547
[1] "non of the conditions"
[1] 82323
[1] "non of the conditions"
[1] 152
[1] 0.09825469
[1] "Hyperlipidemia"
[1] 25280
[1] 1040
[1] 0.6722689
# A tibble: 3 × 3
  person_id condition_concept_id standard_concept_name
      <dbl>                <dbl> <chr>                
1   1053250               432867 Hyperlipidemia       
2   3503647               432867 Hyperlipidemia       
3   1185387               432867 Hyperlipidemia       
[1] "obesity"
[1] 11976
[1] 765
[1] 0.4945055
# A tibble: 3 × 3
  person_id condition_concept_id standard_concept_name
      <dbl>                <dbl> <chr>                
1   9706700               433736 Obesity              
2   3503647               433736 Obesity              
3   3503647               433736 Obesity              
[1] "hypertension"
[1] 42157
[1] 1152
[1] 0.7446671
# A tibble: 3 × 3
  person_id condition_concept_id standard_concept_name 
      <dbl>                <dbl> <chr>                 
1   1958922               320128 E

NULL

In [36]:
# INSOMNIA read data 
library(tidyverse)
library(bigrquery)

insomnia.pid.sql <- paste(insomnia.pid, collapse = ", ")

insomnia_survey_sql <- paste("
    SELECT
        c_occurrence.person_id,
        c_occurrence.condition_concept_id,
        c_standard_concept.concept_name as standard_concept_name,
        c_standard_concept.concept_code as standard_concept_code,
        c_standard_concept.vocabulary_id as standard_vocabulary,
        c_occurrence.condition_start_datetime,
        c_occurrence.condition_end_datetime,
        c_occurrence.condition_type_concept_id,
        c_type.concept_name as condition_type_concept_name,
        c_occurrence.stop_reason,
        c_occurrence.visit_occurrence_id,
        visit.concept_name as visit_occurrence_concept_name,
        c_occurrence.condition_source_value,
        c_occurrence.condition_source_concept_id,
        c_source_concept.concept_name as source_concept_name,
        c_source_concept.concept_code as source_concept_code,
        c_source_concept.vocabulary_id as source_vocabulary,
        c_occurrence.condition_status_source_value,
        c_occurrence.condition_status_concept_id,
        c_status.concept_name as condition_status_concept_name 
    FROM
        ( SELECT
            * 
        FROM
            `condition_occurrence` c_occurrence 
        WHERE
            (
                condition_concept_id IN (SELECT
                    DISTINCT c.concept_id 
                FROM
                    `cb_criteria` c 
                JOIN
                    (SELECT
                        CAST(cr.id as string) AS id       
                    FROM
                        `cb_criteria` cr       
                    WHERE
                        concept_id IN (195771, 201254, 201820, 201826, 4008576, 4024659, 40482801, 40482883, 40484648, 40484649, 40485020, 4058243, 4063042, 4063043, 4099214, 4099334, 4099651, 4129519, 4130162, 4152858, 4193704, 4200875, 4202383, 4230254, 42537681, 4263902, 4274025, 4304377, 4311629, 43531007, 43531008, 43531009, 43531010, 436670, 443238, 443412, 44808385, 45757474, 45757789)       
                        AND full_text LIKE '%_rank1]%'      ) a 
                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                        OR c.path LIKE CONCAT('%.', a.id) 
                        OR c.path LIKE CONCAT(a.id, '.%') 
                        OR c.path = a.id) 
                WHERE
                    is_standard = 1 
                    AND is_selectable = 1)
                )
            AND 
                c_occurrence.person_id IN (", insomnia.pid.sql, ")
            ) c_occurrence 
        LEFT JOIN
            `concept` c_standard_concept 
                ON c_occurrence.condition_concept_id = c_standard_concept.concept_id 
        LEFT JOIN
            `concept` c_type 
                ON c_occurrence.condition_type_concept_id = c_type.concept_id 
        LEFT JOIN
            `visit_occurrence` v 
                ON c_occurrence.visit_occurrence_id = v.visit_occurrence_id 
        LEFT JOIN
            `concept` visit 
                ON v.visit_concept_id = visit.concept_id 
        LEFT JOIN
            `concept` c_source_concept 
                ON c_occurrence.condition_source_concept_id = c_source_concept.concept_id 
        LEFT JOIN
            `concept` c_status 
                ON c_occurrence.condition_status_concept_id = c_status.concept_id", sep="")


# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
insomnia_survey_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "insomnia_survey",
  "insomnia_survey_*.csv")
message(str_glue('The data will be written to {insomnia_survey_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), insomnia_survey_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  insomnia_survey_path,
  destination_format = "CSV")


The data will be written to gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260505/insomnia_survey/insomnia_survey_*.csv. Use this path when reading the data into your notebooks in the future.



In [9]:
library(tidyverse)
library(bigrquery)

read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(standard_concept_name = col_character(), standard_concept_code = col_character(), standard_vocabulary = col_character(), condition_type_concept_name = col_character(), stop_reason = col_character(), visit_occurrence_concept_name = col_character(), condition_source_value = col_character(), source_concept_name = col_character(), source_concept_code = col_character(), source_vocabulary = col_character(), condition_status_source_value = col_character(), condition_status_concept_name = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
insomnia_survey_path <- 'gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260505/insomnia_survey/insomnia_survey_*.csv'

insomnia_survey <- read_bq_export_from_workspace_bucket(insomnia_survey_path)

dim(insomnia_survey)

head(insomnia_survey, 5)

Loading gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260505/insomnia_survey/insomnia_survey_000000000000.csv.



[1] 993821     20

person_id,condition_concept_id,standard_concept_name,standard_concept_code,standard_vocabulary,condition_start_datetime,condition_end_datetime,condition_type_concept_id,condition_type_concept_name,stop_reason,visit_occurrence_id,visit_occurrence_concept_name,condition_source_value,condition_source_concept_id,source_concept_name,source_concept_code,source_vocabulary,condition_status_source_value,condition_status_concept_id,condition_status_concept_name
<dbl>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<chr>,<chr>,<dbl>,<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>,<dbl>,<chr>
1778971,4001450,Lumbago with sciatica,202794004,SNOMED,2022-09-19 17:32:00 UTC,NA,5086,Condition tested for by diagnostic procedure,NA,7e+15,Outpatient Visit,M54.41,45543613,"Lumbago with sciatica, right side",M54.41,ICD10CM,NA,NA,NA
6256904,80809,Rheumatoid arthritis,69896004,SNOMED,2023-01-04 13:40:00 UTC,NA,5086,Condition tested for by diagnostic procedure,NA,7e+15,Outpatient Visit,M06.9,35208753,"Rheumatoid arthritis, unspecified",M06.9,ICD10CM,NA,NA,NA
6256904,438409,Attention deficit hyperactivity disorder,406506008,SNOMED,2020-06-02 11:03:00 UTC,NA,5086,Condition tested for by diagnostic procedure,NA,7e+15,Outpatient Visit,F90.9,35207265,"Attention-deficit hyperactivity disorder, unspecified type",F90.9,ICD10CM,NA,NA,NA
6256904,443211,Benign prostatic hyperplasia with outflow obstruction,236646007,SNOMED,2023-01-23 13:07:00 UTC,NA,5086,Condition tested for by diagnostic procedure,NA,7e+15,Outpatient Visit,N40.1,45601728,Benign prostatic hyperplasia with lower urinary tract symptoms,N40.1,ICD10CM,NA,NA,NA
9004048,437082,Ankylosing spondylitis,9631008,SNOMED,2023-07-20 15:45:00 UTC,NA,5086,Condition tested for by diagnostic procedure,NA,7e+15,Outpatient Visit,M45.9,45582150,Ankylosing spondylitis of unspecified sites in spine,M45.9,ICD10CM,NA,NA,NA


In [38]:
condition_func(df = insomnia_survey)

[1] 1625
[1] "non of the conditions"
[1] 66864
[1] "non of the conditions"
[1] 333
[1] 0.2049231
[1] "Hyperlipidemia"
[1] 21341
[1] 907
[1] 0.5581538
# A tibble: 3 × 3
  person_id condition_concept_id standard_concept_name
      <dbl>                <dbl> <chr>                
1   2316011               432867 Hyperlipidemia       
2   3259490               432867 Hyperlipidemia       
3   1753752               432867 Hyperlipidemia       
[1] "obesity"
[1] 9868
[1] 668
[1] 0.4110769
# A tibble: 3 × 3
  person_id condition_concept_id standard_concept_name
      <dbl>                <dbl> <chr>                
1   2316011               433736 Obesity              
2   3266772               433736 Obesity              
3   3164897               433736 Obesity              
[1] "hypertension"
[1] 33645
[1] 920
[1] 0.5661538
# A tibble: 3 × 3
  person_id condition_concept_id standard_concept_name 
      <dbl>                <dbl> <chr>                 
1   6256904               320128 Essen

NULL

In [39]:
# INSOMNIA read data 
library(tidyverse)
library(bigrquery)

pca.pid.sql <- paste(prostate.pid, collapse = ", ")

pca_survey_sql <- paste("
    SELECT
        c_occurrence.person_id,
        c_occurrence.condition_concept_id,
        c_standard_concept.concept_name as standard_concept_name,
        c_standard_concept.concept_code as standard_concept_code,
        c_standard_concept.vocabulary_id as standard_vocabulary,
        c_occurrence.condition_start_datetime,
        c_occurrence.condition_end_datetime,
        c_occurrence.condition_type_concept_id,
        c_type.concept_name as condition_type_concept_name,
        c_occurrence.stop_reason,
        c_occurrence.visit_occurrence_id,
        visit.concept_name as visit_occurrence_concept_name,
        c_occurrence.condition_source_value,
        c_occurrence.condition_source_concept_id,
        c_source_concept.concept_name as source_concept_name,
        c_source_concept.concept_code as source_concept_code,
        c_source_concept.vocabulary_id as source_vocabulary,
        c_occurrence.condition_status_source_value,
        c_occurrence.condition_status_concept_id,
        c_status.concept_name as condition_status_concept_name 
    FROM
        ( SELECT
            * 
        FROM
            `condition_occurrence` c_occurrence 
        WHERE
            (
                condition_concept_id IN (SELECT
                    DISTINCT c.concept_id 
                FROM
                    `cb_criteria` c 
                JOIN
                    (SELECT
                        CAST(cr.id as string) AS id       
                    FROM
                        `cb_criteria` cr       
                    WHERE
                        concept_id IN (195771, 201254, 201820, 201826, 4008576, 4024659, 40482801, 40482883, 40484648, 40484649, 40485020, 4058243, 4063042, 4063043, 4099214, 4099334, 4099651, 4129519, 4130162, 4152858, 4193704, 4200875, 4202383, 4230254, 42537681, 4263902, 4274025, 4304377, 4311629, 43531007, 43531008, 43531009, 43531010, 436670, 443238, 443412, 44808385, 45757474, 45757789)       
                        AND full_text LIKE '%_rank1]%'      ) a 
                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                        OR c.path LIKE CONCAT('%.', a.id) 
                        OR c.path LIKE CONCAT(a.id, '.%') 
                        OR c.path = a.id) 
                WHERE
                    is_standard = 1 
                    AND is_selectable = 1)
                )
            AND 
                c_occurrence.person_id IN (", pca.pid.sql, ")
            ) c_occurrence 
        LEFT JOIN
            `concept` c_standard_concept 
                ON c_occurrence.condition_concept_id = c_standard_concept.concept_id 
        LEFT JOIN
            `concept` c_type 
                ON c_occurrence.condition_type_concept_id = c_type.concept_id 
        LEFT JOIN
            `visit_occurrence` v 
                ON c_occurrence.visit_occurrence_id = v.visit_occurrence_id 
        LEFT JOIN
            `concept` visit 
                ON v.visit_concept_id = visit.concept_id 
        LEFT JOIN
            `concept` c_source_concept 
                ON c_occurrence.condition_source_concept_id = c_source_concept.concept_id 
        LEFT JOIN
            `concept` c_status 
                ON c_occurrence.condition_status_concept_id = c_status.concept_id", sep="")


# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
pca_survey_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "pca_survey",
  "pca_survey_*.csv")
message(str_glue('The data will be written to {pca_survey_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), pca_survey_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  pca_survey_path,
  destination_format = "CSV")

The data will be written to gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260505/pca_survey/pca_survey_*.csv. Use this path when reading the data into your notebooks in the future.



In [10]:
library(tidyverse)
library(bigrquery)

read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(standard_concept_name = col_character(), standard_concept_code = col_character(), standard_vocabulary = col_character(), condition_type_concept_name = col_character(), stop_reason = col_character(), visit_occurrence_concept_name = col_character(), condition_source_value = col_character(), source_concept_name = col_character(), source_concept_code = col_character(), source_vocabulary = col_character(), condition_status_source_value = col_character(), condition_status_concept_name = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
pca_survey_path <- 'gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260505/pca_survey/pca_survey_*.csv'

pca_survey <- read_bq_export_from_workspace_bucket(pca_survey_path)

dim(pca_survey)

head(pca_survey, 5)

Loading gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260505/pca_survey/pca_survey_000000000000.csv.



[1] 136770     20

person_id,condition_concept_id,standard_concept_name,standard_concept_code,standard_vocabulary,condition_start_datetime,condition_end_datetime,condition_type_concept_id,condition_type_concept_name,stop_reason,visit_occurrence_id,visit_occurrence_concept_name,condition_source_value,condition_source_concept_id,source_concept_name,source_concept_code,source_vocabulary,condition_status_source_value,condition_status_concept_id,condition_status_concept_name
<dbl>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<chr>,<chr>,<dbl>,<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>,<dbl>,<chr>
6360268,4185471,Disorder of pigmentation,414032001,SNOMED,2020-01-28 00:00:00 UTC,NA,44786629,Secondary Condition,NA,3.0e+16,Outpatient Visit,414032001,4185471,Disorder of pigmentation,414032001,SNOMED,NA,NA,NA
9365803,201620,Kidney stone,95570007,SNOMED,2013-09-17 00:00:00 UTC,NA,44786627,Primary Condition,NA,3.0e+16,Inpatient Visit,N20.0,35209281,Calculus of kidney,N20.0,ICD10CM,NA,NA,NA
6844736,135777,Neoplasm of uncertain behavior of skin,95087004,SNOMED,2017-06-27 00:00:00 UTC,NA,32810,Claim,NA,1.3e+16,Office Visit,D48.5,35206687,Neoplasm of uncertain behavior of skin,D48.5,ICD10CM,NA,NA,NA
1901736,138384,Acquired hypothyroidism,111566002,SNOMED,2020-07-04 00:00:00 UTC,NA,44786629,Secondary Condition,NA,5.9e+16,NA,E03.8,35206858,Other specified hypothyroidism,E03.8,ICD10CM,NA,NA,NA
1901736,138384,Acquired hypothyroidism,111566002,SNOMED,2020-07-16 00:00:00 UTC,NA,44786629,Secondary Condition,NA,5.9e+16,NA,E03.8,35206858,Other specified hypothyroidism,E03.8,ICD10CM,NA,NA,NA


In [41]:
condition_func(df = pca_survey)

[1] 281
[1] "non of the conditions"
[1] 13625
[1] "non of the conditions"
[1] 36
[1] 0.1281139
[1] "Hyperlipidemia"
[1] 4237
[1] 192
[1] 0.683274
# A tibble: 3 × 3
  person_id condition_concept_id standard_concept_name
      <dbl>                <dbl> <chr>                
1   1128377               432867 Hyperlipidemia       
2   2042423               432867 Hyperlipidemia       
3   5224273               432867 Hyperlipidemia       
[1] "obesity"
[1] 839
[1] 71
[1] 0.252669
# A tibble: 3 × 3
  person_id condition_concept_id standard_concept_name
      <dbl>                <dbl> <chr>                
1   4913734               433736 Obesity              
2   8702495               433736 Obesity              
3   2332633               433736 Obesity              
[1] "hypertension"
[1] 7914
[1] 199
[1] 0.7081851
# A tibble: 3 × 3
  person_id condition_concept_id standard_concept_name 
      <dbl>                <dbl> <chr>                 
1   9365803               320128 Essential hyp

NULL

In [42]:
# INSOMNIA read data 
library(tidyverse)
library(bigrquery)

healthy.pid.sql <- paste(healthy.pid, collapse = ", ")
length(healthy.pid)

healthy_survey_sql <- paste("
    SELECT
        c_occurrence.person_id,
        c_occurrence.condition_concept_id,
        c_standard_concept.concept_name as standard_concept_name,
        c_standard_concept.concept_code as standard_concept_code,
        c_standard_concept.vocabulary_id as standard_vocabulary,
        c_occurrence.condition_start_datetime,
        c_occurrence.condition_end_datetime,
        c_occurrence.condition_type_concept_id,
        c_type.concept_name as condition_type_concept_name,
        c_occurrence.stop_reason,
        c_occurrence.visit_occurrence_id,
        visit.concept_name as visit_occurrence_concept_name,
        c_occurrence.condition_source_value,
        c_occurrence.condition_source_concept_id,
        c_source_concept.concept_name as source_concept_name,
        c_source_concept.concept_code as source_concept_code,
        c_source_concept.vocabulary_id as source_vocabulary,
        c_occurrence.condition_status_source_value,
        c_occurrence.condition_status_concept_id,
        c_status.concept_name as condition_status_concept_name 
    FROM
        ( SELECT
            * 
        FROM
            `condition_occurrence` c_occurrence 
        WHERE
            (
                condition_concept_id IN (SELECT
                    DISTINCT c.concept_id 
                FROM
                    `cb_criteria` c 
                JOIN
                    (SELECT
                        CAST(cr.id as string) AS id       
                    FROM
                        `cb_criteria` cr       
                    WHERE
                        concept_id IN (195771, 201254, 201820, 201826, 4008576, 4024659, 40482801, 40482883, 40484648, 40484649, 40485020, 4058243, 4063042, 4063043, 4099214, 4099334, 4099651, 4129519, 4130162, 4152858, 4193704, 4200875, 4202383, 4230254, 42537681, 4263902, 4274025, 4304377, 4311629, 43531007, 43531008, 43531009, 43531010, 436670, 443238, 443412, 44808385, 45757474, 45757789)       
                        AND full_text LIKE '%_rank1]%'      ) a 
                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                        OR c.path LIKE CONCAT('%.', a.id) 
                        OR c.path LIKE CONCAT(a.id, '.%') 
                        OR c.path = a.id) 
                WHERE
                    is_standard = 1 
                    AND is_selectable = 1)
                )
            AND 
                c_occurrence.person_id IN (", healthy.pid.sql, ")
            ) c_occurrence 
        LEFT JOIN
            `concept` c_standard_concept 
                ON c_occurrence.condition_concept_id = c_standard_concept.concept_id 
        LEFT JOIN
            `concept` c_type 
                ON c_occurrence.condition_type_concept_id = c_type.concept_id 
        LEFT JOIN
            `visit_occurrence` v 
                ON c_occurrence.visit_occurrence_id = v.visit_occurrence_id 
        LEFT JOIN
            `concept` visit 
                ON v.visit_concept_id = visit.concept_id 
        LEFT JOIN
            `concept` c_source_concept 
                ON c_occurrence.condition_source_concept_id = c_source_concept.concept_id 
        LEFT JOIN
            `concept` c_status 
                ON c_occurrence.condition_status_concept_id = c_status.concept_id", sep="")


# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
healthy_survey_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "healthy_survey",
  "healthy_survey_*.csv")
message(str_glue('The data will be written to {healthy_survey_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), healthy_survey_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  healthy_survey_path,
  destination_format = "CSV")

[1] 6960

The data will be written to gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260505/healthy_survey/healthy_survey_*.csv. Use this path when reading the data into your notebooks in the future.



In [11]:
library(tidyverse)
library(bigrquery)

read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(standard_concept_name = col_character(), standard_concept_code = col_character(), standard_vocabulary = col_character(), condition_type_concept_name = col_character(), stop_reason = col_character(), visit_occurrence_concept_name = col_character(), condition_source_value = col_character(), source_concept_name = col_character(), source_concept_code = col_character(), source_vocabulary = col_character(), condition_status_source_value = col_character(), condition_status_concept_name = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
healthy_survey_path <- 'gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260505/healthy_survey/healthy_survey_*.csv'

healthy_survey <- read_bq_export_from_workspace_bucket(healthy_survey_path)

dim(healthy_survey)

head(healthy_survey, 5)

Loading gs://fc-secure-1d0240db-6bfa-4a11-a526-48cfbb0d664a/bq_exports/zqiu@researchallofus.org/20260505/healthy_survey/healthy_survey_000000000000.csv.



[1]  0 20

person_id,condition_concept_id,standard_concept_name,standard_concept_code,standard_vocabulary,condition_start_datetime,condition_end_datetime,condition_type_concept_id,condition_type_concept_name,stop_reason,visit_occurrence_id,visit_occurrence_concept_name,condition_source_value,condition_source_concept_id,source_concept_name,source_concept_code,source_vocabulary,condition_status_source_value,condition_status_concept_id,condition_status_concept_name
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>


In [44]:
condition_func(df = healthy_survey)

[1] 0
[1] "non of the conditions"
[1] 0
[1] "non of the conditions"
[1] 0
[1] NaN
[1] "Hyperlipidemia"
[1] 0
[1] 0
[1] NaN
# A tibble: 3 × 3
  person_id condition_concept_id standard_concept_name
  <chr>     <chr>                <chr>                
1 NA        NA                   NA                   
2 NA        NA                   NA                   
3 NA        NA                   NA                   
[1] "obesity"
[1] 0
[1] 0
[1] NaN
# A tibble: 3 × 3
  person_id condition_concept_id standard_concept_name
  <chr>     <chr>                <chr>                
1 NA        NA                   NA                   
2 NA        NA                   NA                   
3 NA        NA                   NA                   
[1] "hypertension"
[1] 0
[1] 0
[1] NaN
# A tibble: 3 × 3
  person_id condition_concept_id standard_concept_name
  <chr>     <chr>                <chr>                
1 NA        NA                   NA                   
2 NA        NA                   NA

NULL

In [57]:
diabetes_meta$cohort <- 'Diabetes'
insomnia_meta$cohort <- 'Insomnia'
pca_meta$cohort <- 'Prostate'
healthy_meta$cohort <- 'Healthy'

meta_all <- rbind(diabetes_meta,insomnia_meta,pca_meta,healthy_meta)
meta_all$age <- as.integer(floor(interval(meta_all$date_of_birth, Sys.Date()) / years(1)))
meta_all[1:3,]
table(meta_all$cohort)

person_id,date_of_birth,race_concept_id,race,ethnicity_concept_id,ethnicity,sex_at_birth_concept_id,sex_at_birth,self_reported_category_concept_id,self_reported_category,cohort,age
<dbl>,<chr>,<dbl>,<chr>,<dbl>,<chr>,<dbl>,<chr>,<dbl>,<chr>,<chr>,<int>
1344748,1966-06-15 00:00:00 UTC,2e+09,More than one population,38003563,Hispanic or Latino,45878463,Female,2e+09,More than one population,Diabetes,59
1617237,1964-06-15 00:00:00 UTC,2e+09,More than one population,38003563,Hispanic or Latino,45878463,Female,2e+09,More than one population,Diabetes,61
2977576,1968-06-15 00:00:00 UTC,2e+09,More than one population,38003563,Hispanic or Latino,45878463,Female,2e+09,More than one population,Diabetes,57



Diabetes  Healthy Insomnia Prostate 
    1547     6960     1625      281 

In [13]:
diabetes_survey$cohort <- 'Diabetes'
insomnia_survey$cohort <- 'Insomnia'
pca_survey$cohort <- 'Prostate'
healthy_survey$cohort <- 'Healthy'

survey_all <- rbind(diabetes_survey,insomnia_survey,pca_survey,healthy_survey)
survey_all[1:3,]

person_id,condition_concept_id,standard_concept_name,standard_concept_code,standard_vocabulary,condition_start_datetime,condition_end_datetime,condition_type_concept_id,condition_type_concept_name,stop_reason,⋯,visit_occurrence_concept_name,condition_source_value,condition_source_concept_id,source_concept_name,source_concept_code,source_vocabulary,condition_status_source_value,condition_status_concept_id,condition_status_concept_name,cohort
<dbl>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<chr>,<chr>,⋯,<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>,<dbl>,<chr>,<chr>
9706700,4174262,Polyneuropathy,42345000,SNOMED,2019-05-06 16:29:47 UTC,2019-05-06 16:29:47 UTC,38000280,Observation recorded from EHR,NA,⋯,Outpatient Visit,G62.9,35207458,"Polyneuropathy, unspecified",G62.9,ICD10CM,NA,NA,NA,Diabetes
1053250,437246,Vitamin B deficiency,47903000,SNOMED,2018-06-20 14:56:00 UTC,NA,5086,Condition tested for by diagnostic procedure,NA,⋯,Outpatient Visit,266.2,44829891,Other B-complex deficiencies,266.2,ICD9CM,NA,NA,NA,Diabetes
2100232,257628,Systemic lupus erythematosus,55464009,SNOMED,2020-06-22 05:30:00 UTC,NA,5086,Condition tested for by diagnostic procedure,NA,⋯,Outpatient Visit,M32.9,35208827,"Systemic lupus erythematosus, unspecified",M32.9,ICD10CM,NA,NA,NA,Diabetes


In [66]:
final_demographic_plot_condition_func(df = survey_all, path.out = '/home/jupyter/workspaces/result/pre_final/AcrossCohort/')

ERROR: [1m[33mError[39m in `distinct()`:[22m
[1m[22m[33m![39m Must use existing variables.
[31m✖[39m `age` not found in `.data`.


In [73]:
final_demographic_plot_age_func(df = meta_all, path.out = '/home/jupyter/workspaces/result/pre_final/AcrossCohort/')

NULL

In [64]:
 survey_all %>%
    group_by(cohort, person_id) %>%
    summarise(n_conditions = n_distinct(standard_concept_name	), .groups = "drop") %>%
    group_by(cohort) %>%
    summarise(median_conditions = median(n_conditions), max_conditions = max(n_conditions), .groups = "drop")


cohort,median_conditions,max_conditions
<chr>,<int>,<int>
Diabetes,51,634
Insomnia,58,455
Prostate,43,230


In [14]:
df <- merge (survey_all, meta_all[,c('person_id', 'age')], by = 'person_id', all=TRUE)
survey_cci <- cci_calc_func(df = df, path.out = NULL)
dim(survey_cci)
table(survey_cci$cohort)

[1] 10413     3


Diabetes Insomnia Prostate 
    1547     1625      281 

In [58]:
healthy_cci <- healthy_cci_func(df = meta_all[which(meta_all$cohort == 'Healthy'),])
colnames(healthy_cci)[3] <- 'cci_score'
dim(healthy_cci)
healthy_cci <- healthy_cci[, c('person_id', 'cci_score')]
healthy_cci$cohort <- 'Healthy'
healthy_cci[1:3,]

[1] 6960    3

person_id,cci_score,cohort
<dbl>,<dbl>,<chr>
1000464,1,Healthy
1000883,3,Healthy
1001942,3,Healthy


In [59]:
survey_cci <- survey_cci[!is.na(survey_cci$cohort) & (survey_cci$cohort!= 'Healthy'),]
survey_cci[1:3,]

library(dplyr)
survey_cci <- bind_rows(survey_cci, healthy_cci)
dim(survey_cci)
table(survey_cci$cohort)

cohort,person_id,cci_score
<chr>,<dbl>,<dbl>
Diabetes,1003770,6
Diabetes,1003895,12
Diabetes,1011367,1


[1] 10413     3


Diabetes  Healthy Insomnia Prostate 
    1547     6960     1625      281 

In [60]:
write.table(survey_cci, '/home/jupyter/workspaces/result/pre_final/pre_final_cci.txt', sep = '\t', quote = F, row.names = F)


In [61]:

final_demographic_plot_cii_func(survey_cci, path.out = '/home/jupyter/workspaces/result/pre_final/AcrossCohort/')

# A tibble: 14 × 6
   cohort   cci_category   n_cohort median_cci n_category   pct
   <chr>    <fct>             <int>      <dbl>      <int> <dbl>
 1 Diabetes None (0)           1547          5         19   1.2
 2 Diabetes Mild (1-2)         1547          5        190  12.3
 3 Diabetes Moderate (3-4)     1547          5        379  24.5
 4 Diabetes Severe (≥5)        1547          5        959  62  
 5 Healthy  None (0)           6960          1       2480  35.6
 6 Healthy  Mild (1-2)         6960          1       2491  35.8
 7 Healthy  Moderate (3-4)     6960          1       1989  28.6
 8 Insomnia None (0)           1625          4        161   9.9
 9 Insomnia Mild (1-2)         1625          4        374  23  
10 Insomnia Moderate (3-4)     1625          4        481  29.6
11 Insomnia Severe (≥5)        1625          4        609  37.5
12 Prostate Mild (1-2)          281          5         16   5.7
13 Prostate Moderate (3-4)      281          5        123  43.8
14 Prostate Severe (≥

NULL

In [63]:
# subset fimal pid
summary(survey_cci)
dim(survey_cci)
final_pid <- survey_cci[which(survey_cci$cci_score <= 10), c('cohort', 'person_id')];
final_pid <- rbind(final_pid)
dim(final_pid)

    cohort            person_id         cci_score     
 Length:10413       Min.   :1000464   Min.   : 0.000  
 Class :character   1st Qu.:2107936   1st Qu.: 0.000  
 Mode  :character   Median :3828013   Median : 2.000  
                    Mean   :4573109   Mean   : 2.661  
                    3rd Qu.:6911215   3rd Qu.: 4.000  
                    Max.   :9997301   Max.   :20.000  

[1] 10413     3

[1] 10165     2

In [35]:
write.table(final_pid, '/home/jupyter/workspaces/result/final/final_pid.txt', sep = '\t', quote = F, row.names = F)
